### Load a dataset from the Hub

- Dataset's metadata/general info is stored  inside DatasetInfo
    - Holds description, features, size
- load_dataset_builder() loads a dataset builder
    - dataset builder - object with metadata/configuration info w/o loading data itslef
    - loads a preview of data but doesn't actually download it


In [ ]:
from datasets import load_dataset_builder
ds_builder = load_dataset_builder("cornell-movie-review-data/rotten_tomatoes")

#View description and features
print(ds_builder.info.description)
print(ds_builder.info.features)

#Once you're sure you're happy with dataset, load it!
from datasets import load_dataset 

#SPLITS
#We can specify which split of dataset we want: train/validation/test 
dataset = load_dataset('rotten_tomatoes', split = 'train')

#We can explicity view the split names with get_dataset_split_names()
from datasets import get_dataset_split_names
print(get_dataset_split_names("cornell-movie-review-data/rotten_tomatoes"))

#If you don't specify a split, you'll get a DatasetDict object
    #Dictionary where each key is the name, value is its dataset split
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
dataset

- CONFIGRATIONS
- Some datsets contain several sub-datasets
    - ex.) MInDS-14 has sub-datasets for audio data in different languages
    - Sub-datasets = configurations or subsets
- Configurations must be explicitly selected with 

In [ ]:
#Can view list of configurations (subsets) with get_dataset_config_names()
from datasets import get_dataset_config_names
configs = get_dataset_config_names('PolyAI/minds14', trust_remote_code=True)
print(configs)

#Then pick the config you want, let's load French version
mindsFR = load_dataset("PolyAI/minds14", "fr-FR", split="train", trust_remote_code=True)

- REMOTE CODE
- Some datasets repositories have a specific loading script
- A loading script is a Python script that defines how to load and prepare a dataset ...
    - ...from its source (e.g., files, URLs, or APIs).
- Loading scripts shows...
    - Where to get the data (e.g., download URLs, files).
    - How to process it into a Hugging Face Dataset object.
    - How to define the features (e.g., text, labels, integers).
- usually needed for custom datasets that aren't in a simple CSV, JSON, or text format.

In [ ]:
#trust_remote_code = True needed to use dataset with loading script
from datasets import get_dataset_config_names, get_dataset_split_names, load_dataset

#dataset = load_dataset("allenai/c4", "en", split="train", trust_remote_code=True)
#configs = get_dataset_config_names("allenai/c4", trust_remote_code=True)
#splits = get_dataset_split_names("allenai/c4", "en", trust_remote_code=True)
#configs, splits
print('Dataset loaded')

### Know your dataset

- Two types of dataset objects: 1. Dataset (regular/default), 2. IterableDataset
- Dataset: fast random access to rows, memory-mapping to load big datasets w/ small memory
- IterableDataset: for humongous datasets (can't fit on desk/in-memory). Can access/use dataset w/o waiting for complete download
- Key terms
    - Fast random access: You can instantly access any row by index (e.g., dataset[100]).
    - Memory-mapping: data is stored on disk, doesn\u2019t load everything into RAM at once.
- INDEXING
    - Can acess different rows/cols of dataset
    - Negative indexing works
    - Can index cols: dataset['column_name']
    - Order matters
    - Slicing is the same as with pandas

In [ ]:
#DATASET object - many features!
#Let's work with an example
from datasets import load_dataset
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train")

#INDEXING 
# Can acess different rows/cols of dataset
# Negative indexing works
# Can index cols: dataset['column_name']
print(dataset[0])
print(dataset[-1])
#print(dataset['text']) --> this gets a whole lot of text
#can combine row and cols: dataset[row_number][column_name]
dataset[0]["text"]

#Order Indexing Matters
#You could also do: dataset[column_name][row_number]
    #Accessing columns then row takes much longer 
    #row indexing is faster in general
import time
start_time = time.time()
text = dataset[0]["text"]
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

start_time = time.time()
text = dataset["text"][0]
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

#SLICING
#We can use : to get subset of data 
#First three rows:
print(dataset[:3])
#rows 3-6 not including 6
print(dataset[3:6])

- ITERABLE DATASETS
- IterableDataset - loaded when you set streaming parameter to Ture in load_dataset
    - Data is continously streamed in, one row at a time
    - Data is downloaded on the fly instead of all at once 
    - Immediate use!
- You can create an IterableDataset from an existing Dataset object using .to_iterable_dataset()
- Cons:
    - No random accessing/indexing
    - Instead must iterate over elements to get next item with next(iter())
- We can get a subset using .take()

In [ ]:
from datasets import load_dataset
iterable_dataset = load_dataset('food101', split = 'train', streaming = True)
for example in iterable_dataset:
    print(example)
    break

#You can also create an IterableDataset from an existing Dataset object
    #use .to_iterable_dataset() 
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train")
iterable_dataset = dataset.to_iterable_dataset()

#Cons to IterableDatasets
    #No random accessing/indexing
#Instead, must iterate over elements to get next item with next(iter())
next(iter(iterable_dataset))

#We can get subset using IterableDataset.take() to get next n examples
list(iterable_dataset.take(3))

#Difference b/w Dataset and Iterable Dataset conceptual guide
    #Process Guide: preprocess a Dataset. Stream guide to learn how preprocess IterableDataset.

### Preprocessing

- Huggingface datasets allow for loading and preprocessing 
    - Very diverse set of preprocessing functions to get dataset into approproiate format
- Basic function examples:
    - rename column, 
    - unflatted nested fields
- Most common ML preprocessing needs:
    - Tokenize a text dataset
    - Resample an audio dataset
    - Apply transforms to an image dataset 
- Last preprocessing step is ...
    - Making sure dataset format is compatible with ML framework's input 

- HOW TO PREPROCESS TEXT: TOKENIZE TEXT
- Converting words into tokens to then be converted to numbers/embeddings 
- Makes sure to use same tokenizer as pretrained model 
    - When you\u2019re passing new inputs (like movie reviews from Rotten Tomatoes) ...
    - ... into a pretrained model like BERT, 
    - you need to process the inputs the same way that BERT was trained \u2014
    - so the model can understand them.
- Tokenizer returns a dict of 3 items:
    - input_ids: numbers representing tokens in text - each token gets unique number from model vocabulary
    - token_type_ids indicate which sequence a token belongs to - 0 or 1 for question, answer sequence
    - attention_mask: indicates wheteher a token should be masked or not - 
              - a 1 means model should pay attention to this token
              - 0 = padding = ignore. Adds extra inputs to make all inputs same length
- These three things are what a model uses

- Fastest way to tokenize entire dataset: map() function
    - Speeds up process by applying tokenizer to batches instead of individual examples

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train")

# Call tokenizer on first row of text
tokenizer(dataset[0]['text']
# Tokenizer returns a dict of 3 items:
    #input_ids: numbers representing tokens in text - each token gets unique number from model vocabulary
    #token_type_ids indicate which sequence a token belongs to - 0 or 1 for question, answer sequence
    #attention_mask: indicates wheteher a token should be masked or not - 
              # a 1 means model should pay attention to this token
              # 0 = padding = ignore. Adds extra inputs to make all inputs same length
# These three things are what a model uses

# Fastest way to tokenize entire dataset: map() function
    # Speeds up process by applying tokenizer to batches instead of individual examples
def tokenization(example):
    return toenizer(example['text'])
dataset = dataset.map(tokenization), batched = True)

#Lastly, make sure dataset is compatabile with ML framework
#ex.) set_format() to make compatible with PyTorch
dataset.set_format(type="torch", columns=["input_ids", "token_type_ids", "attention_mask", "label"])
dataset.format['type']

- HOW TO PREPROCESS AUDIO: RESAMPLE AUDIO SIGNALS
- Audio must also be divided into discrete data points --> "sampling"
    - Sampling rate = tells you how of speech signal captured per second
- Sampling rate of dataset must match sampling rate as your model (whatever data was used to pretrain the model)
- example:
    - dataset: MInDS-14 is a spoken language understanding dataset that contains spoken commands in multiple languages.
    - Audio feature import: lets you load and work with audio data in Hugging Face Datasets
        - tells dataset to decode audio files when you load rows 
    - feature extractor corresponding to pretrianed Wav2Vec2 Model
        - feature extractor - turn audio waveforms into numerical features for model to understand
        - takes raw audio waveform transforms, converts into features
        - basically word tokenizer but for audio 

In [ ]:

from transformers import AutoFeatureExtractor
from datasets import load_dataset, Audio

feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
dataset = load_dataset("PolyAI/minds14", "en-US", split="train")

# Indexing to audio column --> automatically decoded and resampled
dataset[0]['audio']

#Look at dataset card and model card to determine sampling rate + compatability 
#We can resample dataset to match sampling rate with cast_column(0 from Audio feature
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
dataset[0]["audio"]

#Then we can use map() function to resample entire dataset as needed --> batch processing is faster again!
def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays, sampling_rate=feature_extractor.sampling_rate, max_length=16000, truncation=True
    )
    return inputs

dataset = dataset.map(preprocess_function, batched=True)

- HOW TO PREPROCESS IMAGES: APPLY DATA AUGMENTATIONS TO IMAGES
- Data augmentation - process of introducing random variations to image without changing meaning of data
    - ex.) changing colors, randomly cropping
- Huggingface Datasets can help!

In [ ]:
#Start by loading the Beans dataset, the Image feature, and the feature extractor corresponding to a pretrained ViT model:
from transformers import AutoFeatureExtractor
from datasets import load_dataset, Image

feature_extractor = AutoFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k")
dataset = load_dataset("AI-Lab-Makerere/beans", split="train")

# Index into the first row of the dataset. When you call the image column of the dataset, the underlying PIL object is automatically decoded into an image.
dataset[0]["image"]

#Most image models expect the image to be in the RGB mode. The Beans images are already in the RGB mode, but if your dataset contains images in a different mode, you can use the cast_column() function to set the mode to RGB:
dataset = dataset.cast_column("image", Image(mode="RGB"))

# Now apply whateber transforms you want. Experiment w/ what's available in torchvision
# ex.) randomly rotating an image:
from torchvision.transforms import RandomRotation
rotate = RandomRotation(degrees=(0, 90))
def transforms(examples):
    examples["pixel_values"] = [rotate(image) for image in examples["image"]]
    return examples

#set_transform() function applies transform on-the-fly
    # when you index into image, transorm is applied and image gets rotated 
dataset.set_transform(transforms)
dataset[0]["pixel_values"]


### Create a Dataset

- We can manually create our own Dataset object to work with our own data
- Then we get all the advantages of Hugging Face datasets:
    - fast loading
    - stream processing for enormous datasets 
    - memory-mapping: data is stored on disk, doesn\u2019t load everything into RAM at once.
        - load data from disk into memory "on demand" without actually loading entire dataset into RAM all at once
        - photocopying entire book into hands vs leaving book on desk and reading pages as needed
- How to do it?
    - First, low-code approaches. Can often be as easy as dragging and dropping data files into dataset repo on the Hub
- Low-code methods
    - Folder-based builders for quickly creating image/audio dataset
    - from_ methods for creating datasets from local files

- METHOD 1: FILE-BASED BUILDERS
    - Many common formats supported: csv,json,parquet,txt
    - ex) Make Dataset from one/list of CSV files
- METHOD 2: FOLDER-BASED BUILDERS
    - Two options: ImageFolder, AudioFolder --> create image/audio dataset quickly w/ several thousand examples
        - Good for protyping computer vision/speech models before scaling to a larger dataset
    - How they work: take data, automatically generate dataset's features, splits, labels
        - ImageFolder: uses image feature which decodes image file. Supports common image extension formats
        - AudioFolder: uses audio feature which decodes audio file. Supports extensions such as wav and mp3. 
        - Splits generated from repository structure. 
        - Labels inferred from directory name
        - ex.) pokemon/train/grass/bulbasaur.png --> train = split, grass = label, png = image to decode
- METHOD 3: FROM PYTHON DICTIONARIES
    - Two ways to create data using from_ methods
    - 1) from_generator() 
        - memory efficient b/c of generators iterative behavior
        - useful for really large datases that may not fit in memory 
        - dataset is generated on disk progressively, then memory-mapped
    - 2) rom_dict() - straightforward way to create dataset from dict
        - for audio/image dataset. Can use cast_column() with from_dict9) to speciffy column and feature type
#ex.) audio
- NOTE: Can also include metadata (text captions/transcriptions) 
    - make metadata.csv file in folder. Have it correspond with file_name column. 
    - format: file_name, text
    - ex.) charmander.png, It has a preference for hot things.

In [ ]:
#METHOD 1: FILE-BASED BUILDERS
from datasets import load_dataset
dataset = load_dataset('csv', data_files = 'my_files.csv')

#METHOD 2: FOLDER-BASED BUILDERS
#Two options: ImageFolder, AudioFolder --> create image/audio dataset quickly w/ several thousand examples

#ex.) Create image dataset from imagefolder 
from datasets import load_dataset
dataset = load_dataset("imagefolder", data_dir="/path/to/pokemon")

#ex.) Same thing for audio
from datasets import load_dataset
dataset = load_dataset("audiofolder", data_dir="/path/to/folder")

# Can also include metadata (text captions/transcriptions) 
    # make metadata.csv file in folder. Have it correspond with file_name column. 
    #format: file_name, text
    #ex.) charmander.png, It has a preference for hot things.
#METHOD 3: FROM PYTHON DICTIONARIES
# Two ways to create data using from_ methods

# 1. from_generator() 
    # memory efficient b/c of generators iterative behavior
    # useful for really large datases that may not fit in memory 
        # dataset is generated on disk progressively, then memory-mapped
from datasets import Dataset
def gen():
    yield {"pokemon": "bulbasaur", "type": "grass"}
    yield {"pokemon": "squirtle", "type": "water"}
ds = Dataset.from_generator(gen)
ds[0]

#A generator-based IterableDataset needs to be iterated over with a for loop
from datasets import Dataset
def gen():
    yield {"pokemon": "bulbasaur", "type": "grass"}
    yield {"pokemon": "squirtle", "type": "water"}
ds = Dataset.from_generator(gen)
ds[0]

#2. from_dict() - straightforward way to create dataset from dict
from datasets import Dataset
ds = Dataset.from_dict({"pokemon": ["bulbasaur", "squirtle"], "type": ["grass", "water"]})
ds[0]

#or audio/image dataset. Can use cast_column() with from_dict9) to speciffy column and feature type
#ex.) audio
audio_dataset = Dataset.from_dict({"audio": ["path/to/audio_1", ..., "path/to/audio_n"]}).cast_column("audio", Audio())


### Share a Dataset to the Hub